# Mission 2 - wav2vec2 신고자/119대원 화자 분류 학습 (Colab)

사용 순서:
1. SMOKE_MODE = True 로 먼저 소규모 전체 흐름 검증 (파이프라인/모델/loss 정상 여부, 몇 분 내 끝남)
2. 본 데이터 shard zip 업로드 후 SMOKE_MODE = False 로 본 학습
3. 이 노트북은 실행 로그(각 epoch loss/accuracy)가 남아있는 상태로 제출해야 함

Mission1(wav2vec2 성별 분류) 노트북과 파이프라인 구조는 동일하고,
라벨(gender -> speaker)과 평가 단위(call -> segment)만 다르다.
Mission2는 한 통화 안에 speaker=0/1이 항상 같이 섞여 있어서
(대화 양쪽 다 있음) call 단위가 아니라 **발화 조각(segment) 단위**로
평가/제출해야 한다는 점이 Mission1과 가장 큰 차이.

## 0. 환경 설정

In [ ]:
!pip install -q transformers torchaudio soundfile


In [ ]:
SMOKE_MODE = False  # 먼저 True로 파이프라인부터 검증 후 False로 전환

# Drive의 03_pack_for_colab.py 결과물 업로드 경로 (팀/본인 상황에 맞게 수정)
DRIVE_UPLOAD_DIR = "/content/drive/MyDrive/dcc_mission2/upload"

# 로컬(Colab 세션) 작업 디렉토리 - Drive 직접 접근은 느리므로
# 항상 로컬 디스크로 복사해서 사용한다
LOCAL_DATA_DIR = "/content/data"
LOCAL_WAV_DIR = f"{LOCAL_DATA_DIR}/wav"

CHECKPOINT_NAME = "mission2_wav2vec2_best.pt"
DRIVE_CKPT_DIR = "/content/drive/MyDrive/dcc_mission2/checkpoints"

# 사전학습 체크포인트
# Mission1이 large(kresnik, 3.15억 파라미터)는 1 epoch에 12시간+
# 걸리는 걸 확인하고 base(9400만 파라미터, Zeroth-Korean으로
# 한국어 사전학습)로 교체해서 속도를 확보했음 -> 동일하게 재사용
PRETRAINED_CKPT = "Kkonjeong/wav2vec2-base-korean"

NUM_LABELS = 2


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 데이터 로컬 다운로드 + 압축 해제

- SMOKE_MODE=True 이면 `smoke/smoke.zip` + `smoke/smoke_chunk_manifest.csv` 사용
- SMOKE_MODE=False 이면 `shards/shard_*.zip` 전부 + `manifest/chunk_manifest.csv` 사용
- wav2vec2_common.py 도 함께 복사해서 로컬/Colab에서 동일 상수/함수 사용

In [ ]:
import os
import shutil
import zipfile
import glob

os.makedirs(LOCAL_WAV_DIR, exist_ok=True)

# 세션 런타임(로컬)은 재연결하면 이미 받은 데이터가 사라질 수 있으므로
# 중복다운로드를 피한다. 단, SMOKE_MODE를 True->False로 바꾼 경우처럼
# "이전에 받은 것과 지금 원하는 게 다른 상황"이 될 수 있어서
# 마커 파일로 어떤 모드로 받았는지 같이 확인한다.
MODE_MARKER_PATH = f"{LOCAL_DATA_DIR}/.download_mode"
current_mode = "smoke" if SMOKE_MODE else "full"

already_present = (
    os.path.exists(MODE_MARKER_PATH)
    and open(MODE_MARKER_PATH).read().strip() == current_mode
    and len(os.listdir(LOCAL_WAV_DIR)) > 0
)

if already_present:
    print(f"이미 '{current_mode}' 데이터가 로컬에 존재 - 다운로드/압축해제 건너뜀")
    print("wav 파일 수:", len(os.listdir(LOCAL_WAV_DIR)))

else:
    def copy_and_unzip(zip_path, dest_dir):
        local_zip = f"/content/_tmp_{os.path.basename(zip_path)}"
        shutil.copy2(zip_path, local_zip)
        with zipfile.ZipFile(local_zip, "r") as zf:
            zf.extractall(dest_dir)
        os.remove(local_zip)

    if SMOKE_MODE:
        copy_and_unzip(
            f"{DRIVE_UPLOAD_DIR}/smoke/smoke.zip",
            LOCAL_WAV_DIR
        )
        shutil.copy2(
            f"{DRIVE_UPLOAD_DIR}/smoke/smoke_chunk_manifest.csv",
            f"{LOCAL_DATA_DIR}/chunk_manifest.csv"
        )
    else:
        shard_zips = sorted(glob.glob(f"{DRIVE_UPLOAD_DIR}/shards/shard_*.zip"))
        print("shard 개수:", len(shard_zips))
        for i, zip_path in enumerate(shard_zips, start=1):
            print(f"[{i}/{len(shard_zips)}] {zip_path}")
            copy_and_unzip(zip_path, LOCAL_WAV_DIR)
        shutil.copy2(
            f"{DRIVE_UPLOAD_DIR}/manifest/chunk_manifest.csv",
            f"{LOCAL_DATA_DIR}/chunk_manifest.csv"
        )

    with open(MODE_MARKER_PATH, "w") as f:
        f.write(current_mode)

    print("wav 파일 수:", len(os.listdir(LOCAL_WAV_DIR)))


shard 개수: 15
[1/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_00.zip
[2/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_01.zip
[3/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_02.zip
[4/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_03.zip
[5/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_04.zip
[6/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_05.zip
[7/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_06.zip
[8/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_07.zip
[9/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_08.zip
[10/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_09.zip
[11/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_10.zip
[12/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_11.zip
[13/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_12.zip
[14/15] /content/drive/MyDrive/dcc_mission2/upload/shards/shard_13.zip
[1

In [ ]:
# wav2vec2_common.py 를 Code/ 에서 그대로 Drive에도 올려두고 복사해서 사용
# (train / inference 양쪽에서 MIN_DUR_SEC / MAX_DUR_SEC / compute_chunks 등
#  같은 기준을 써야 하므로 상수를 따로따로 적지 않고 이 파일에서 import 한다)
shutil.copy2(
    f"{DRIVE_UPLOAD_DIR}/wav2vec2_common.py",
    "/content/wav2vec2_common.py"
)

import sys
sys.path.append("/content")

from wav2vec2_common import (
    TARGET_SR,
    MIN_DUR_SEC
)


## 2. Dataset

- chunk_manifest.csv의 (segment_file, chunk_start_sec, chunk_end_sec) 정보로 로드
- 8kHz 원본을 16kHz로 on-the-fly 리샘플
- MIN_DUR_SEC보다 짧으면 그 길이까지 zero-padding
- call_id / segment_file을 함께 반환 (val에서 **segment 단위** 집계에 사용 -
  Mission1과 달리 call 단위가 아님. 한 통화 안에 speaker=0/1이 항상 섞여
  있으므로 call_id로 묶으면 서로 다른 라벨의 segment가 섞여버림)

In [ ]:
import csv
import os
import torch
import torchaudio
from torch.utils.data import Dataset


class SpeakerSegmentChunkDataset(Dataset):

    def __init__(self, manifest_path, wav_dir, split):
        self.wav_dir = wav_dir

        # 로컬에 실제로 존재하는 wav만 사용
        # (shard 업로드 일부만 받은 상태로 스모크 테스트하는 경우
        #  등을 위해, 없는 파일은 건너뛰고 있는 파일만으로 학습/평가한다.
        #  특정 화자/구간만 누락되는 게 아니라 shard(=call_id 단위 묶음)
        #  단위로 누락되므로 라벨이 한쪽으로 쏠릴 위험은 크지 않음.)
        available_files = set(os.listdir(wav_dir))

        self.rows = []
        skipped = 0

        with open(manifest_path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["split"] != split:
                    continue
                if row["segment_file"] not in available_files:
                    skipped += 1
                    continue
                self.rows.append(row)

        print(f"[{split}] 대상 청크: {len(self.rows)}개, "
              f"로컬에 없어서 건너뜀: {skipped}개")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        wav_path = f"{self.wav_dir}/{row['segment_file']}"
        waveform, orig_sr = torchaudio.load(wav_path)

        # 항상 mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # 청크 구간 slicing (원본 sr 기준)
        start = int(float(row["chunk_start_sec"]) * orig_sr)
        end = int(float(row["chunk_end_sec"]) * orig_sr)
        waveform = waveform[:, start:end]

        # 8kHz -> 16kHz 리샘플
        if orig_sr != TARGET_SR:
            waveform = torchaudio.functional.resample(
                waveform, orig_sr, TARGET_SR
            )

        # 최소 길이 zero-padding
        min_samples = int(MIN_DUR_SEC * TARGET_SR)
        if waveform.shape[1] < min_samples:
            pad = min_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad))

        # speaker는 이미 0/1 정수라 별도 매핑(GENDER_TO_LABEL 같은 것) 불필요
        label = int(row["speaker"])

        return {
            "input_values": waveform.squeeze(0).numpy(),
            "label": label,
            "call_id": row["call_id"],
            "segment_file": row["segment_file"]
        }


## 3. Collate (배치 시 dynamic padding + attention_mask)

In [ ]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(PRETRAINED_CKPT)


def collate_fn(batch):
    input_values = [item["input_values"] for item in batch]

    features = feature_extractor(
        input_values,
        sampling_rate=TARGET_SR,
        padding=True,
        return_tensors="pt"
    )

    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    call_ids = [item["call_id"] for item in batch]
    segment_files = [item["segment_file"] for item in batch]

    return {
        "input_values": features["input_values"],
        # base(group-norm) 계열 wav2vec2 feature extractor는
        # attention_mask를 아예 안 만들어도 됨 (base 모델은
        # 원래 attention_mask 없이 학습하는 게 HF 공식 권장 사항)
        "attention_mask": features.get("attention_mask"),
        "labels": labels,
        "call_ids": call_ids,
        "segment_files": segment_files
    }


## 4. DataLoader 구성

In [ ]:
from torch.utils.data import DataLoader

MANIFEST_PATH = f"{LOCAL_DATA_DIR}/chunk_manifest.csv"

train_dataset = SpeakerSegmentChunkDataset(MANIFEST_PATH, LOCAL_WAV_DIR, "train")
val_dataset = SpeakerSegmentChunkDataset(MANIFEST_PATH, LOCAL_WAV_DIR, "val")

print("train 청크 수:", len(train_dataset))
print("val 청크 수:", len(val_dataset))

BATCH_SIZE = 16 if not SMOKE_MODE else 4

# 오디오 리샘플/슬라이싱이 CPU에서 일어나므로, A100처럼 GPU가 빠르면
# 데이터 로딩이 병목이 되기 쉬움. num_workers를 넉넉히 줘서 GPU를
# 놀리지 않게 한다 (Colab 코어 수에 따라 4~8 사이에서 조정 가능).
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS
)


[train] 대상 청크: 717537개, 로컬에 없어서 건너뜀: 0개
[val] 대상 청크: 180989개, 로컬에 없어서 건너뜀: 0개
train 청크 수: 717537
val 청크 수: 180989


## 5. 모델 로드 + freeze 구성

- feature encoder(CNN)는 항상 freeze
- transformer는 마지막 N개 layer만 학습 (나머지는 freeze) - 시간 여유되면 N을 늘려서 실험

In [ ]:
import torch.nn as nn
from transformers import Wav2Vec2ForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    PRETRAINED_CKPT, num_labels=NUM_LABELS
)
model.freeze_feature_encoder()

# 실험2의 컨텍스트 마진(±0.3초)이 대회 규정 위반으로 확인되어 폐기.
# 실험2에서 같이 바꿨던 N_UNFROZEN_LAYERS(2->4)만 분리해서
# 실험1의 데이터 파이프라인(마진 없는 세그먼트 그대로) 위에 재실험.
N_UNFROZEN_LAYERS = 4  # 마지막 4개 transformer layer만 학습

total_layers = len(model.wav2vec2.encoder.layers)
for i, layer in enumerate(model.wav2vec2.encoder.layers):
    requires_grad = (i >= total_layers - N_UNFROZEN_LAYERS)
    for p in layer.parameters():
        p.requires_grad = requires_grad

model.to(device)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"학습 대상 파라미터: {n_trainable:,} / 전체 {n_total:,}")


device: cuda


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: Kkonjeong/wav2vec2-base-korean
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
projector.weight  | MISSING    | 
classifier.weight | MISSING    | 
projector.bias    | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


학습 대상 파라미터: 33,665,666 / 전체 94,569,090


## 6. Optimizer / Scheduler

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# 스모크는 2epoch만: 1epoch만으로는 head가 아직 초기화 상태여서
# 다음 몇 에폭에서 학습이 정상인지 판단하기 어려움
NUM_EPOCHS = 2 if SMOKE_MODE else 4

# head(분류기, 새로 초기화된 레이어)는 backbone보다 훨씬 큰 LR이 필요함.
# 낮은 LR로 두면 head가 거의 안 움직여서 loss가 ln(2) 근처에 머무름.
BACKBONE_LR = 2e-5
HEAD_LR = 1e-3

head_params = []
backbone_params = []

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith(("classifier", "projector", "layer_weights")):
        head_params.append(p)
    else:
        backbone_params.append(p)

print(f"head 파라미터: {sum(p.numel() for p in head_params):,}")
print(f"backbone(unfrozen) 파라미터: {sum(p.numel() for p in backbone_params):,}")

optimizer = AdamW([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": head_params, "lr": HEAD_LR}
])

total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# train split의 speaker 분포로 class weight 계산
# (0=119대원 48% / 1=신고자 52% 정도로 거의 균형이지만, train subset
#  기준으로 다시 계산해서 사용)
from collections import Counter
train_speaker_counts = Counter(row["speaker"] for row in train_dataset.rows)
total_train = sum(train_speaker_counts.values())
class_weight = torch.tensor([
    total_train / (2 * train_speaker_counts["0"]),
    total_train / (2 * train_speaker_counts["1"])
], dtype=torch.float32).to(device)
print("class_weight (speaker=0, speaker=1):", class_weight.tolist())

loss_fn = nn.CrossEntropyLoss(weight=class_weight)


head 파라미터: 197,378
backbone(unfrozen) 파라미터: 33,468,288
class_weight (speaker=0, speaker=1): [1.0463048219680786, 0.9576199650764465]


## 7. Segment 단위 평가 함수

Mission 2 최종 목표는 **발화 조각(segment) 1개당 1개 라벨**이므로
(Mission1처럼 통화 1개당 1개가 아님), 청크 단위 accuracy가 아니라
같은 segment_file에서 나온 청크들의 확률을 평균 낸 뒤 argmax로
segment 단위 최종 예측을 정한다. (한 segment가 MAX_DUR_SEC보다 길면
여러 청크로 나뉘어 있으므로 이 집계가 필요함 - 대부분은 청크가 1개뿐이라
평균이 곧 그 청크의 확률과 같음)

In [ ]:
import numpy as np
from collections import defaultdict


@torch.no_grad()
def evaluate(loader):
    model.eval()

    segment_probs = defaultdict(list)
    segment_label = {}

    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"]
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(
                input_values=input_values,
                attention_mask=attention_mask
            )
            loss = loss_fn(outputs.logits, labels)

        total_loss += loss.item()
        n_batches += 1

        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

        for segment_file, label, prob in zip(batch["segment_files"], batch["labels"].numpy(), probs):
            segment_probs[segment_file].append(prob)
            segment_label[segment_file] = label

    correct = 0
    for segment_file, probs in segment_probs.items():
        mean_prob = np.mean(probs, axis=0)
        pred = int(np.argmax(mean_prob))
        if pred == segment_label[segment_file]:
            correct += 1

    segment_level_acc = correct / len(segment_probs)
    avg_loss = total_loss / n_batches

    return avg_loss, segment_level_acc


## 8. 학습 루프 (로그 출력 - 저장한 .ipynb에 남아있어야 함)

In [ ]:
import time

best_val_acc = 0.0

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

# 컴퓨팅 단위 예산이 빠듯할 수 있어서, 실측 속도 기반으로 전체 예상
# 소요 시간(ETA)을 로그에 남긴다. 이거 보고 필요하면 NUM_EPOCHS를
# 줄이거나 중간에 런타임을 끊을지 판단할 것.
train_run_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    epoch_start = time.time()

    for step, batch in enumerate(train_loader, start=1):
        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"]
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(
                input_values=input_values,
                attention_mask=attention_mask
            )
            loss = loss_fn(outputs.logits, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()

        if step % 50 == 0:
            elapsed = time.time() - train_run_start
            steps_done_total = (epoch - 1) * len(train_loader) + step
            steps_per_sec = steps_done_total / elapsed
            remaining_steps = total_steps - steps_done_total
            eta_hours = (remaining_steps / steps_per_sec) / 3600 if steps_per_sec > 0 else float("nan")

            print(f"epoch {epoch} step {step}/{len(train_loader)} "
                  f"train_loss={running_loss / step:.4f} "
                  f"steps/sec={steps_per_sec:.2f} "
                  f"train_ETA={eta_hours:.2f}h (평가 시간 별도)")

    val_loss, val_segment_acc = evaluate(val_loader)
    print(f"[epoch {epoch}] train_loss={running_loss / len(train_loader):.4f} "
          f"val_loss={val_loss:.4f} val_segment_level_acc={val_segment_acc:.4f} "
          f"epoch_elapsed={(time.time() - epoch_start) / 60:.1f}min")

    if val_segment_acc > best_val_acc:
        best_val_acc = val_segment_acc
        os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "pretrained_ckpt": PRETRAINED_CKPT,
                "num_labels": NUM_LABELS,
                "val_segment_level_acc": val_segment_acc
            },
            f"{DRIVE_CKPT_DIR}/{CHECKPOINT_NAME}"
        )
        print(f"  -> best checkpoint 저장 (val_segment_level_acc={val_segment_acc:.4f})")

total_elapsed_hours = (time.time() - train_run_start) / 3600
print(f"\n학습 종료. best val segment-level accuracy: {best_val_acc}")
print(f"총 소요 시간: {total_elapsed_hours:.2f}시간")


epoch 1 step 50/44847 train_loss=0.6917 steps/sec=11.14 train_ETA=4.47h (평가 시간 별도)
epoch 1 step 100/44847 train_loss=0.6910 steps/sec=12.72 train_ETA=3.91h (평가 시간 별도)
epoch 1 step 150/44847 train_loss=0.6905 steps/sec=13.21 train_ETA=3.77h (평가 시간 별도)
epoch 1 step 200/44847 train_loss=0.6904 steps/sec=13.57 train_ETA=3.67h (평가 시간 별도)
epoch 1 step 250/44847 train_loss=0.6901 steps/sec=13.91 train_ETA=3.58h (평가 시간 별도)
epoch 1 step 300/44847 train_loss=0.6897 steps/sec=14.14 train_ETA=3.52h (평가 시간 별도)
epoch 1 step 350/44847 train_loss=0.6886 steps/sec=14.35 train_ETA=3.47h (평가 시간 별도)
epoch 1 step 400/44847 train_loss=0.6874 steps/sec=14.50 train_ETA=3.43h (평가 시간 별도)
epoch 1 step 450/44847 train_loss=0.6858 steps/sec=14.60 train_ETA=3.40h (평가 시간 별도)
epoch 1 step 500/44847 train_loss=0.6841 steps/sec=14.77 train_ETA=3.37h (평가 시간 별도)
epoch 1 step 550/44847 train_loss=0.6816 steps/sec=14.89 train_ETA=3.34h (평가 시간 별도)
epoch 1 step 600/44847 train_loss=0.6791 steps/sec=14.97 train_ETA=3.32h (평가 

## 마지막: 다음 단계

- SMOKE_MODE=True로 이 노트북 전체를 실행해서 파이프라인 정상 동작, loss가 줄어드는지 확인
- 스모크 통과 후 `03_pack_for_colab.py`가 만든 `shards/`, `manifest/`를 Drive에 업로드
- SMOKE_MODE=False로 바꾸고 NUM_EPOCHS / N_UNFROZEN_LAYERS 조정해서 본 학습
- 학습 끝나면 이 .ipynb를 (로그 남은 상태로) 그대로 제출 파일 중 하나로 사용
- 체크포인트(`DRIVE_CKPT_DIR`)를 로컬로 받아서 inference.py에서 사용
  (inference.py는 startAt/endAt/speaker만으로 원본 wav에서 직접 발화를 잘라내야
  하므로, 이 노트북의 학습 파이프라인과 동일한 리샘플/청크 기준을
  wav2vec2_common.py를 통해 그대로 재사용해야 함)